# Phase 1: Baseline Training Run (EfficientNet-B0 + Cross-Entropy)

This notebook executes **Phase 1** of our implementation plan:
- Backbone: **EfficientNet-B0** (pretrained on ImageNet)
- Head: Standard linear classification head (Softmax)
- Loss: **Cross-Entropy Loss** with inverse-frequency class weights
- Optimizer: AdamW (`lr=3e-4`, `weight_decay=1e-4`)
- Scheduler: `CosineAnnealingWarmRestarts` (`T_0=10`)
- Primary Metric: Validation **Quadratic Weighted Kappa (QWK)** with Early Stopping (patience=7)
- Checkpoint Saved: `checkpoints/ce_baseline_best.pth`

### Step 1: Verify Colab GPU & Working Directory

In [ ]:
import os
import torch

print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Active GPU: {torch.cuda.get_device_name(0)}")
print(f"Current Working Directory: {os.getcwd()}")

### Step 2: Check or Download APTOS 2019 Dataset
The dataset must be placed in `data/aptos2019/`:
- `data/aptos2019/train.csv`
- `data/aptos2019/train_images/*.png`

In [ ]:
import os

data_csv = 'data/aptos2019/train.csv'
images_dir = 'data/aptos2019/train_images'

if os.path.exists(data_csv) and os.path.exists(images_dir):
    num_images = len(os.listdir(images_dir))
    print(f"✅ APTOS dataset found! {num_images} images ready.")
else:
    print("❌ Dataset not found at data/aptos2019/")
    print("To download via Kaggle API, uncomment and run the Kaggle download block below.")

In [ ]:
# === Optional: Automated Kaggle API Download ===
# 1. Upload your kaggle.json to the Files panel in Colab
# 2. Uncomment and run the lines below:

# !mkdir -p ~/.kaggle
# !cp kaggle.json ~/.kaggle/
# !chmod 600 ~/.kaggle/kaggle.json
# !mkdir -p data/aptos2019
# !kaggle competitions download -c aptos2019-blindness-detection -p data/aptos2019/
# !unzip -q data/aptos2019/aptos2019-blindness-detection.zip -d data/aptos2019/

### Step 3: Run Baseline Training (Phase 1)
This runs the baseline model. Training progress, loss curves, and validation QWK are logged in real-time.

In [ ]:
!python scripts/train.py loss=cross_entropy model=efficientnet_b0 training=baseline wandb.enabled=true

### Step 4: Display Baseline Results Table
Reads and displays the saved baseline performance table.

In [ ]:
import pandas as pd

results_path = "results/tables/efficientnet_b0_cross_entropy.csv"
if os.path.exists(results_path):
    df_res = pd.read_csv(results_path)
    print("\n⭐ Phase 1 Baseline Results:")
    display(df_res)
else:
    print(f"{results_path} will be created after training finishes.")